# NB2_CPT_NoPPL_Colab
HF project: `boods/FrMedQA-CrossLingual-v2-NoPPL-<qlora|bf16>-*`
Path: `/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2`


In [ ]:
# ⚙️ Stage 0 — Install dependencies (Colab)
import subprocess, sys, importlib

REQUIRED = {"unsloth":"unsloth","transformers":"transformers","peft":"peft",
            "bitsandbytes":"bitsandbytes","datasets":"datasets",
            "rouge_score":"rouge-score","nltk":"nltk","bert_score":"bert-score",
            "huggingface_hub":"huggingface-hub","tqdm":"tqdm",
            "sklearn":"scikit-learn","sacrebleu":"sacrebleu", "modelscope": "modelscope",
            "datasketch": "datasketch"}

missing_pkgs_to_install = []
for mod, pkg in REQUIRED.items():
    if importlib.util.find_spec(mod) is None:
        missing_pkgs_to_install.append(pkg)

if missing_pkgs_to_install:
    print(f"Attempting to install missing packages: {missing_pkgs_to_install}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs_to_install)
        print("Successfully installed missing packages. A kernel restart is recommended for full effect.")
    except subprocess.CalledProcessError as e:
        print(f"Error during installation of {missing_pkgs_to_install}: {e}. Please install manually.")
        raise
    except Exception as e:
        print(f"An unexpected error occurred during package installation: {e}")
        raise

try:
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)
except Exception:
    pass
print("✓ Stage 0 done")


In [ ]:
# ⚙️ Stage 0b — Drive mount, paths, secrets
import os, sys

def _on_colab():
    try:
        import google.colab; return True
    except Exception: return False

if _on_colab():
    from google.colab import drive
    drive.mount("/content/drive")

    def _secret(name, prompt):
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v: return v
        except Exception: pass
        import getpass
        return getpass.getpass(prompt)
    os.environ["HF_TOKEN"] = _secret("HF_TOKEN", "HF_TOKEN: ")
    _wb = _secret("WANDB_API_KEY", "WANDB_API_KEY (blank to skip): ")
    if _wb:
        os.environ["WANDB_API_KEY"] = _wb
    else:
        os.environ["WANDB_DISABLED"] = "true"

BASE_DIR = os.environ.get("FRMEDQA_BASE", "/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2")
REPO_DIR    = BASE_DIR
RESULTS_DIR = os.path.join(BASE_DIR, "RESULT")
EVALS_DIR   = os.path.join(BASE_DIR, "EVALS")
CACHE_DIR   = os.path.join(EVALS_DIR, "results_cache")
OUT_DIR     = os.path.join(BASE_DIR, "analysis_output")
FIG_DIR     = os.path.join(OUT_DIR, "figures")
for d in (RESULTS_DIR, EVALS_DIR, CACHE_DIR, OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["FRMEDQA_REPO"] = REPO_DIR
os.environ["FRMEDQA_BASE"] = BASE_DIR
os.environ["FRMEDQA_HF_USER"] = "boods"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for v in ("HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE"):
    os.environ[v] = os.path.join(BASE_DIR, "hf_cache")

print(f"BASE_DIR: {BASE_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")


In [ ]:
os.environ["FRMEDQA_PROJECT_NAME"] = "FrMedQA-CrossLingual-v2-NoPPL"
if os.environ.get("FRMEDQA_SEED"):   # cluster multi-seed run: one HF namespace per seed
    os.environ["FRMEDQA_PROJECT_NAME"] += f"-s{os.environ['FRMEDQA_SEED']}"
print(f"HF project: {os.environ['FRMEDQA_PROJECT_NAME']}")

# ─── ONE precision for the whole project (single source of truth) ────────
# qlora = 4-bit NF4 frozen base + bf16 LoRA (QLoRA)   | fits L4 22 GB / A100 40 GB
# bf16  = bf16 frozen base + bf16 LoRA (16-bit LoRA) | needs A100 80 GB / H100
# To switch, edit PRECISION.txt on Drive; every notebook reads the same file.
_pf = os.path.join(BASE_DIR, "PRECISION.txt")
if not os.path.exists(_pf):
    open(_pf, "w").write("qlora")
os.environ["FRMEDQA_PRECISION"] = open(_pf).read().strip().lower()
assert os.environ["FRMEDQA_PRECISION"] in ("qlora", "bf16"), "PRECISION.txt must contain qlora or bf16"
print(f"Precision: {os.environ['FRMEDQA_PRECISION']}  (from {_pf})")


# NB2 — CPT Simple (No Perplexity)
Harvest → clean → lexical density filter → dedup → leakage shield → train.
No PPL scoring, no DSIR, no curriculum ordering. Straightforward CPT.
Pushes to `boods/FrMedQA-CrossLingual-v2-NoPPL-<qlora|bf16>-DAPT`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Stage 1 — import shared modules (cluster-ready)
#   Code (enmed_core / enmed_recipe) comes from the cloned repo + `poetry install`.
#   DATA & OUTPUTS live under $FRMEDQA_BASE (point it at scratch). Code != data.
import os, sys
BASE_DIR = os.environ.get("FRMEDQA_BASE", os.path.expanduser("~/frmedqa_experiments"))
def _find_repo_on_path():
    try:
        import enmed_core  # already importable (cwd = repo root, or installed)
        return
    except ModuleNotFoundError:
        for cand in [os.environ.get("FRMEDQA_REPO"), os.getcwd(),
                     os.path.abspath(os.path.join(os.getcwd(), ".."))]:
            if cand and os.path.exists(os.path.join(cand, "enmed_core.py")):
                sys.path.insert(0, cand); return
        raise ModuleNotFoundError("enmed_core not found — run `poetry install` in the repo, launch Jupyter from the repo root, or set $FRMEDQA_REPO to the repo path.")
_find_repo_on_path()
import enmed_core as ec
import enmed_recipe as er
print("✓ enmed_recipe imported")
print(f"✓ enmed_core imported | BASE_DIR = {BASE_DIR}")

In [ ]:
# Stage 1b — Config / logger / seed / auth
FULL = os.environ.get("FRMEDQA_PROFILE", "colab") == "full"   # cluster: full-scale settings
SEED = int(os.environ.get("FRMEDQA_SEED", "42"))
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=42,
    cpt_lr=2e-5, cpt_epochs=1, cpt_batch_size=2, cpt_grad_accum=(8 if FULL else 4),
    cpt_lora_r=32, cpt_lora_alpha=64, max_seq_length=(2048 if FULL else 1024),
)
cfg.make_dirs()
logger = ec.setup_logger("nb2", log_file=os.path.join(cfg.training_dir, f"nb2_cpt_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)
HF_USER  = cfg.hf_user
USE_WANDB = ec.bootstrap_wandb(project=cfg.wandb_project, logger=logger)

DAPT_OUT_DIR = os.path.join(cfg.training_dir, f"qwen3-14b-cpt-noppl-v2-{cfg.precision}")  # v2: fresh folder, no stale resume
os.makedirs(DAPT_OUT_DIR, exist_ok=True)
cfg.save()
logger.info("="*70)
logger.info(f"NB2 — CPT (SOTA recipe) | base={cfg.base_model}")
logger.info(f"                         | output={DAPT_OUT_DIR}")
logger.info(f"                         | repo={cfg.hf_repo_dapt}")
logger.info("="*70)

In [ ]:
# Full-scale budgets (cluster): same budget for both pipelines, so Perplexity vs NoPPL
# differ only in WHICH documents fill it. Override with FRMEDQA_TOKEN_BUDGET_M="fr,en,gen".
_BUDGET_M = [int(x) for x in os.environ.get("FRMEDQA_TOKEN_BUDGET_M", "250,80,40").split(",")]
# Stage 1c — RECIPE knobs (single source of truth for the data pipeline)
# Set TOKEN_BUDGET=None to consume the corpora in full. The defaults below are
# sized for a single A100; raise/lower per your GPU + wall-clock budget.
RECIPE = {
    # ── token budgets per bucket (None = take everything) ──────────────────
    "fr_bio_token_budget":  (_BUDGET_M[0] * 1_000_000 if FULL else 25_000_000),   # FR biomedical (the domain target)
    "en_bio_token_budget":  (_BUDGET_M[1] * 1_000_000 if FULL else 8_000_000),   # EN biomedical replay
    "gen_token_budget":     (_BUDGET_M[2] * 1_000_000 if FULL else 4_000_000),   # FR+EN general replay
    "max_doc_chars": 6000,

    # ── length gate (exact tokens) ─────────────────────────────────────────
    "min_tokens": 24, "max_tokens": cfg.max_seq_length,

    # ── dedup + leakage ────────────────────────────────────────────────────
    "dedup_threshold": 0.85, "num_perm": 128,
    "leak_threshold": cfg.minhash_threshold,   # 0.8, matches NB1

    # ── perplexity filter (scored on a SAMPLE for thresholds, full for sort) ─
    "ppl_max_len": 512, "ppl_batch_size": 8,
    "ppl_score_cap": (None if FULL else 120_000),   # cap docs sent to PPL scoring (cost guard); None=all
    "ppl_min": 3.0, "ppl_max": 200.0,

    # ── DSIR importance resampling ─────────────────────────────────────────
    "dsir_ngram": 2, "dsir_keep_frac": 0.70,   # keep top 70% most in-domain

    # ── curriculum ─────────────────────────────────────────────────────────
    "curriculum_scheme": "bucketed", "curriculum_buckets": 6,

    # ── replay interleave ──────────────────────────────────────────────────
    "replay_ratio": 0.25,        # 25% of final corpus is replay text

    # ── forgetting probe size ──────────────────────────────────────────────
    "probe_n": 400,
}
import json as _json
logger.info("RECIPE:\n" + _json.dumps(RECIPE, indent=2, default=str))
STAGE_COUNTS = {}   # filled as the funnel progresses; written to manifest at the end

In [ ]:
# Stage 2 — Load Qwen3-14B at cfg.precision (Drive-cached). Loaded BEFORE filtering so the
# same base model both (a) scores document perplexity and (b) gets CPT-trained.
import torch, os
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
for v in ("HF_HOME","TRANSFORMERS_CACHE","HUGGINGFACE_HUB_CACHE","HF_HUB_CACHE"):
    os.environ[v] = cfg.cache_dir
os.makedirs(cfg.cache_dir, exist_ok=True)

# Precision is set by cfg.precision ("qlora" = 4-bit NF4 base + bf16 LoRA,
# "bf16" = bf16 base + bf16 LoRA). The base is always the 16-bit checkpoint.
ec.check_vram_for_precision(cfg, logger=logger)
model, tokenizer = ec.load_model(cfg, cfg.base_model, logger=logger)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
logger.info(f"  loaded | params≈{sum(p.numel() for p in model.parameters())/1e9:.1f}B "
            f"| vocab={len(tokenizer):,}")

In [ ]:
# Stage 3a — FR biomedical sources (the domain we are adapting to)
FR_BIO_SOURCES = [
    {"path": cfg.parcomed_id, "split": "train"},                 # PARCOMED (primary)
    {"path": "DrBenchmark/CAS", "trust_remote_code": True},      # French clinical cases
    {"path": "Dr-BERT/CAS", "trust_remote_code": True},          # mirror
    {"path": "Dr-BERT/QUAERO", "name": "medline", "trust_remote_code": True},
    {"path": "Dr-BERT/QUAERO", "name": "emea",    "trust_remote_code": True},
    {"path": "Dr-BERT/ClinicalCases_FR", "trust_remote_code": True},
    {"path": "bio-datasets/medal"},                              # FR abbrev disambig
]
fr_bio = er.harvest_streaming(
    FR_BIO_SOURCES, token_budget=RECIPE["fr_bio_token_budget"],
    max_doc_chars=RECIPE["max_doc_chars"], logger=logger)
STAGE_COUNTS["harvest_fr_bio"] = len(fr_bio)

In [ ]:
# Stage 3b — EN biomedical replay
EN_BIO_SOURCES = [
    {"path": "ccdv/pubmed-summarization", "name": "document", "split": "train"},
]
en_bio = er.harvest_streaming(
    EN_BIO_SOURCES, token_budget=RECIPE["en_bio_token_budget"],
    max_doc_chars=RECIPE["max_doc_chars"], logger=logger)
STAGE_COUNTS["harvest_en_bio"] = len(en_bio)

In [ ]:
# Stage 3c — General replay (FR + EN Wikipedia) — anti-catastrophic-forgetting
half = (RECIPE["gen_token_budget"] // 2) if RECIPE["gen_token_budget"] else None
GEN_FR = [{"path": "wikimedia/wikipedia", "name": "20231101.fr", "split": "train"}]
GEN_EN = [{"path": "wikimedia/wikipedia", "name": "20231101.en", "split": "train"}]
gen = er.harvest_streaming(GEN_FR, token_budget=half, max_doc_chars=RECIPE["max_doc_chars"], logger=logger)
gen += er.harvest_streaming(GEN_EN, token_budget=half, max_doc_chars=RECIPE["max_doc_chars"], logger=logger)
STAGE_COUNTS["harvest_general"] = len(gen)
replay_pool = en_bio + gen
logger.info(f"✓ Harvest | FR_bio={len(fr_bio):,}  EN_bio={len(en_bio):,}  gen={len(gen):,}")

In [ ]:
# Stage 4 — clean + exact-token length gate (domain bucket)
fr_bio = [er.clean_text(t) for t in fr_bio]
fr_bio = [t for t in fr_bio if t]
fr_bio = er.length_gate(fr_bio, tokenizer,
                        min_tokens=RECIPE["min_tokens"], max_tokens=RECIPE["max_tokens"],
                        logger=logger)
STAGE_COUNTS["clean_length_gate"] = len(fr_bio)

In [ ]:
# Stage 5a — within-corpus MinHash dedup (domain bucket)
fr_bio = er.minhash_dedup(fr_bio, threshold=RECIPE["dedup_threshold"],
                          num_perm=RECIPE["num_perm"], logger=logger)
STAGE_COUNTS["dedup"] = len(fr_bio)

In [ ]:
# Stage 5b — leakage shield: drop any CPT doc overlapping the locked test sets.
# Reuses NB1's test.jsonl fingerprints so CPT can never see test questions.
TEST_JSONL = os.path.join(cfg.results_dir, "test.jsonl")
if os.path.exists(TEST_JSONL):
    test_fps = er.load_test_fingerprints_from_jsonl(TEST_JSONL)
    test_lsh = er.build_test_lsh(test_fps, threshold=RECIPE["leak_threshold"],
                                 num_perm=RECIPE["num_perm"])
    fr_bio = er.leakage_filter(fr_bio, test_lsh, num_perm=RECIPE["num_perm"], logger=logger)
else:
    logger.warning(f"  test.jsonl not found at {TEST_JSONL} — run NB1 first. "
                   "Skipping leakage shield (NOT recommended for the paper).")
STAGE_COUNTS["leakage_shield"] = len(fr_bio)

## 📐 Stage 9 — Replay interleave + build dataset
Spread EN-bio + general replay evenly through the domain corpus (random order, no curriculum).

In [ ]:
# Stage 9 — interleave replay + build dataset (no curriculum, just shuffle)
import random

# replay_pool already exists from Stage 3c (en_bio + gen Wikipedia)
replay_clean = [er.clean_text(t) for t in replay_pool if er.clean_text(t).strip()]
replay_clean = [t for t in replay_clean if len(t) > 200]
replay_clean = er.minhash_dedup(replay_clean, threshold=RECIPE["dedup_threshold"], logger=logger)

# Interleave: domain chunks + replay, then shuffle
all_texts = list(fr_bio) + replay_clean
random.seed(cfg.seed)
random.shuffle(all_texts)
logger.info(f"  corpus: {len(fr_bio):,} domain + {len(replay_clean):,} replay = {len(all_texts):,} total")
STAGE_COUNTS["final_corpus"] = len(all_texts)

# 2% validation split
from datasets import Dataset as HFDataset
val_frac = 0.02
val_n = max(1, int(len(all_texts) * val_frac))
val_texts = all_texts[:val_n]
train_texts = all_texts[val_n:]

train_ds = HFDataset.from_dict({"text": train_texts})
val_ds   = HFDataset.from_dict({"text": val_texts})
logger.info(f"  train={len(train_ds):,}  val={len(val_ds):,}")


## 🏋️ Stage 13 — Attach LoRA + train (curriculum-preserving, resumable)

In [ ]:
# Stage 13a — LoRA adapter
model = FastLanguageModel.get_peft_model(
    model, r=cfg.cpt_lora_r, lora_alpha=cfg.cpt_lora_alpha, lora_dropout=0.0,
    target_modules=cfg.sft_lora_targets, bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg.seed, use_rslora=False, loftq_config=None,
)
model.print_trainable_parameters()

## 🏋️ Stage 10 — Train (standard SFTTrainer, resumable)

In [ ]:
# Stage 10 — standard SFTTrainer (no curriculum, no PPL ordering)
import math
from trl import SFTConfig, SFTTrainer

total_steps = math.ceil(len(train_ds) / (cfg.cpt_batch_size * cfg.cpt_grad_accum))
warmup = max(1, int(total_steps * 0.05))
save_steps = max(1, total_steps // 5)
logger.info(f"CPT plan | steps={total_steps:,}  warmup={warmup}  save_every={save_steps}")

sft_cfg = SFTConfig(
    output_dir=DAPT_OUT_DIR,
    max_seq_length=cfg.max_seq_length,
    packing=True,
    num_train_epochs=cfg.cpt_epochs,
    per_device_train_batch_size=cfg.cpt_batch_size,
    gradient_accumulation_steps=cfg.cpt_grad_accum,
    learning_rate=cfg.cpt_lr,
    lr_scheduler_type="cosine",
    warmup_steps=warmup,
    weight_decay=0.01,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=max(1, total_steps // 50),
    save_steps=save_steps,
    save_total_limit=3,
    seed=cfg.seed,
    report_to="wandb" if USE_WANDB else "none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds,
    args=sft_cfg,
)

result = ec.resumable_train(trainer, DAPT_OUT_DIR, logger=logger)
model.save_pretrained(DAPT_OUT_DIR); tokenizer.save_pretrained(DAPT_OUT_DIR); ec.drive_flush()
logger.info(f"  ✓ final DAPT adapter saved → {DAPT_OUT_DIR}")
ppl_train = math.exp(result.training_loss) if result.training_loss < 20 else float("inf")
logger.info(f"  train_loss={result.training_loss:.4f}  train_ppl={ppl_train:.2f}")


## 📊 Stage 11 — Metrics + manifest

In [ ]:
# Stage 11 — save metrics
ec.atomic_write_json({
    "train_loss": float(result.training_loss),
    "train_ppl": float(ppl_train),
    "corpus_size": len(train_ds),
    "val_size": len(val_ds),
    "stage_counts": STAGE_COUNTS,
}, os.path.join(cfg.results_dir, "nb2_noppl_metrics.json"))
logger.info("  metrics saved")


## 📤 Stage 15 — Push CPT adapter to HF Hub

In [ ]:
# Stage 15 — upload to {HF_USER}/EnToFrMedicaLLM-DAPT
if HF_TOKEN:
    try:
        model.push_to_hub(cfg.hf_repo_dapt, token=HF_TOKEN)
        tokenizer.push_to_hub(cfg.hf_repo_dapt, token=HF_TOKEN)
        logger.info(f"✓ CPT adapter → https://huggingface.co/{cfg.hf_repo_dapt}")
    except Exception as e:
        logger.error(f"HF push failed: {e}")
else:
    logger.warning("• Skipping HF push (no token)")

## 🏁 Stage 16 — Done. Open NB3_SFT.ipynb.

In [ ]:
print("="*70)
print(f"  NB2 (CPT Simple, NoPPL) COMPLETE — {ec.now_ts()}")
print("="*70)
print(f"  Local : {DAPT_OUT_DIR}")
print(f"  HF    : {cfg.hf_repo_dapt}")
print(f"  Train : loss={result.training_loss:.4f}  ppl={ppl_train:.2f}")
print(f"  Corpus: {len(train_ds):,} train + {len(val_ds):,} val")
print("="*70)
print("  -> Open NB3_SFT_NoPPL_Colab.ipynb")


## Stage 16 — Merge DAPT LoRA into the bf16 base
The adapter is merged into the **original 16-bit weights** on CPU (never into 4-bit weights).
NB3 and NB5b load this merged checkpoint at `cfg.precision`. Needs about 30 GB CPU RAM (Colab High-RAM).

In [ ]:
# Stage 16 — Merge the DAPT LoRA into the ORIGINAL bf16 base, on CPU.
# NB3 / NB5b load the merged 16-bit model and quantize it on load (qlora)
# or keep it in bf16 (bf16). A LoRA is never merged into 4-bit weights.
# By default NB3 / NB5b read the merged model straight from Drive, so no
# upload is needed. Set PUSH_MERGED_TO_HUB = True to also publish it.
import gc, torch, psutil
PUSH_MERGED_TO_HUB = False

MERGED_DIR = DAPT_OUT_DIR + "-merged-bf16"
ram_gb = psutil.virtual_memory().total / 1024**3
if ram_gb < 40:
    logger.warning(f"  only {ram_gb:.0f} GB RAM; merging a 14B model needs ~30 GB. "
                   f"Use Runtime > Change runtime type > High-RAM.")

def _find_dapt_adapter():
    """Final DAPT adapter, in order of trust:
    1) saved in DAPT_OUT_DIR itself (end of training),
    2) pushed to the Hub (cfg.hf_repo_dapt, also the final adapter),
    3) the latest trainer checkpoint on Drive (may miss the last steps)."""
    import glob, re as _re
    if os.path.exists(os.path.join(DAPT_OUT_DIR, "adapter_config.json")):
        return DAPT_OUT_DIR, "final adapter on Drive"
    try:
        from huggingface_hub import file_exists
        if HF_TOKEN and file_exists(cfg.hf_repo_dapt, "adapter_config.json", token=HF_TOKEN):
            return cfg.hf_repo_dapt, "final adapter on the Hub"
    except Exception as e:
        logger.warning(f"  Hub check failed: {e}")
    ckpts = [d for d in glob.glob(os.path.join(DAPT_OUT_DIR, "checkpoint-*"))
             if os.path.exists(os.path.join(d, "adapter_config.json"))]
    if ckpts:
        last = max(ckpts, key=lambda d: int(_re.search(r"checkpoint-(\d+)", d).group(1)))
        logger.warning(f"  using the latest trainer checkpoint ({os.path.basename(last)}); "
                       f"steps after it are not included")
        return last, "latest checkpoint"
    raise FileNotFoundError(
        f"No DAPT adapter found in {DAPT_OUT_DIR}, its checkpoints, or {cfg.hf_repo_dapt}.")

def _on_hub(repo):
    try:
        from huggingface_hub import file_exists
        return file_exists(repo, "config.json", token=HF_TOKEN)
    except Exception:
        return False

def _push_folder(folder, repo):
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo, exist_ok=True, private=True)
    api.upload_large_folder(repo_id=repo, folder_path=folder, repo_type="model")  # resumable
    logger.info(f"  ✓ pushed → https://huggingface.co/{repo}")

# 1) Merge (skipped if the merged checkpoint is already on Drive)
if os.path.exists(os.path.join(MERGED_DIR, "config.json")):
    logger.info(f"  merged checkpoint already exists → {MERGED_DIR} (delete the folder to redo)")
else:
    for _name in ("trainer", "model"):
        if _name in globals(): del globals()[_name]
    gc.collect(); torch.cuda.empty_cache()
    adapter_src, where = _find_dapt_adapter()
    logger.info(f"  adapter source: {adapter_src}  ({where})")
    ec.merge_lora_into_bf16(cfg.base_model, adapter_src, MERGED_DIR, push_repo=None,
                            cache_dir=cfg.cache_dir, logger=logger)
    ec.drive_flush()

# 2) Optional: publish to the Hub (resumable; safe to re-run after a disconnect)
if PUSH_MERGED_TO_HUB:
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN is missing.")
    if _on_hub(cfg.hf_repo_dapt_merged):
        logger.info(f"  already on the Hub → https://huggingface.co/{cfg.hf_repo_dapt_merged}")
    else:
        _push_folder(MERGED_DIR, cfg.hf_repo_dapt_merged)

logger.info(f"NB3 / NB5b will load: {MERGED_DIR}  (precision={cfg.precision})")
